# Multi-task GNN Training — Mps1 + Aurora B + hERG

## Purpose
Retrain the Chemprop D-MPNN GNN with multi-task learning across
three targets simultaneously to expand the applicability domain
and enable simultaneous selectivity and safety prediction.

## Tasks
| Task | Target | Role | N compounds |
|------|--------|------|-------------|
| mps1_pIC50 | Mps1/TTK | Primary efficacy | 3,607 |
| aurora_pIC50 | Aurora B (AURKB) | Selectivity | 1,309 |
| herg_pIC50 | hERG (KCNH2) | Cardiac safety | 4,000 |

Missing values are handled with masked loss — compounds without
data for a task do not contribute to that task's loss.

## Rationale
Single-task GNN (Mps1 only) showed low confidence (σ≥0.2) for
77.9% of Phase 2B candidates due to narrow applicability domain.
Multi-task learning expands the domain by incorporating structurally
diverse kinase inhibitors and hERG data.

## Input files
- `multitask_data.zip` — train/val/test CSVs
  Generated by: `kinase_domain/scripts/prepare_multitask_data.py`
  which uses:
    - `kinase_domain/analysis/ic50/expanded_mps1_activity.csv`
    - `kinase_domain/analysis/ic50/aurora_b_activity.csv`
    - `kinase_domain/analysis/ic50/herg_activity.csv`

## Hyperparameters
Same as optimised single-task model (depth=6, hidden_dim=600,
FFN_dim=1700, dropout=0.05) — validated by Ray Tune hpopt.

## Output
- `chemprop_multitask/` — 5 ensemble models
- Predicts: Mps1 pIC50, Aurora B pIC50, hERG pIC50 simultaneously

## To reproduce
1. Run `kinase_domain/scripts/get_multitask_data.py`
2. Run `kinase_domain/scripts/prepare_multitask_data.py`
3. Upload `multitask_data.zip` to Colab
4. Run this notebook (requires T4 GPU, ~45-60 min)

# Cell 1 — Mount Drive and setup

In [ ]:
# Cell 1 — Fixed with full path
from google.colab import drive
import subprocess
import os

drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/mps1-drug-discovery'

# Install Chemprop
subprocess.run(['pip', 'install', 'chemprop', '-q'], capture_output=True)

# Verify with full path
CHEMPROP = '/usr/local/bin/chemprop'
result = subprocess.run(
    [CHEMPROP, '--version'],
    capture_output=True, text=True
)
print(f"Chemprop: {result.stdout.strip() or result.stderr.strip()}")
print(f"Ready!")

Mounted at /content/drive
Chemprop: usage: chemprop [-h] {train,predict,convert,fingerprint,hpopt} ...
chemprop: error: the following arguments are required: mode
Ready!


# Cell 2 — Upload training data


In [ ]:
from google.colab import files
import zipfile

print("Upload multitask_data.zip")
uploaded = files.upload()

with zipfile.ZipFile('multitask_data.zip') as z:
    z.extractall('/content/multitask')

import os
files_extracted = os.listdir('/content/multitask')
print(f"Extracted: {files_extracted}")

Upload multitask_data.zip


Saving multitask_data.zip to multitask_data.zip
Extracted: ['multitask_test.csv', 'multitask_val.csv', 'multitask_train.csv']


# Cell 3 — Train multi-task GNN


In [ ]:
import subprocess, os
from pathlib import Path

CHEMPROP = '/usr/local/bin/chemprop'
TRAIN = '/content/multitask/multitask_train.csv'
VAL   = '/content/multitask/multitask_val.csv'
TEST  = '/content/multitask/multitask_test.csv'
SAVE  = f"{PROJECT}/chemprop_multitask"
os.makedirs(SAVE, exist_ok=True)

cmd = [
    CHEMPROP, 'train',
    '-i', TRAIN, VAL, TEST,
    '-o', SAVE,
    '--task-type',          'regression',
    '--target-columns',     'mps1_pIC50', 'aurora_pIC50', 'herg_pIC50',
    '--epochs',             '50',
    '--batch-size',         '64',
    '--message-hidden-dim', '600',
    '--depth',              '6',
    '--dropout',            '0.05',
    '--ffn-hidden-dim',     '1700',
    '--ffn-num-layers',     '2',
    '--aggregation',        'norm',
    '--aggregation-norm',   '100',
    '--activation',         'RELU',
    '--warmup-epochs',      '12',
    '--max-lr',             '0.0033484567276065727',
    '--init-lr',            '0.00050783769216109',
    '--final-lr',           '7.537471962398345e-05',
    '--num-workers',        '2',
    '--accelerator',        'gpu',
    '--devices',            '1',
    '--ensemble-size',      '5',
    '--metrics',            'rmse', 'r2', 'mae',
]

print("Training multi-task GNN...")
print("Tasks: Mps1 + Aurora B + hERG")
print("5 models × 50 epochs — ~45-60 min\n")

result = subprocess.run(cmd, capture_output=True, text=True)
print("STDOUT (last 3000 chars):")
print(result.stdout[-3000:])
print("\nSTDERR (last 500 chars):")
print(result.stderr[-500:])
print(f"\nReturn code: {result.returncode}")

Training multi-task GNN...
Tasks: Mps1 + Aurora B + hERG
5 models × 50 epochs — ~45-60 min

STDOUT (last 3000 chars):
───────┴────────┴───────┴───────┘
Trainable params: 4.7 M                                                         
Non-trainable params: 0                                                         
Total params: 4.7 M                                                             
Total estimated model params size (MB): 18.936                                  
Modules in train mode: 29                                                       
Modules in eval mode: 0                                                         
Total FLOPs: 0                                                                  
Epoch 49/49 ━━━━━━━━━━━━━━━━ 111/111 0:00:04 •       28.19it/s v_num: 0.000     
                                     0:00:00                   train_loss_step: 
                                                               0.022 val_loss:  
                                      

# Cell 4 — Download the model


In [ ]:
import shutil
shutil.make_archive(
    '/content/chemprop_multitask',
    'zip',
    model_dir
)

from google.colab import files
files.download('/content/chemprop_multitask.zip')
print("Done!")

NameError: name 'model_dir' is not defined